**公开阅读版：** 不包含微博数据集，也不保留运行输出。课堂数据由教师另行提供，放在 `tutorials/data/posts_two_weeks.parquet`。Cookie和采集结果只保留在自己电脑。

# Plotting

**课堂讲解与运行演示用。** 沿用 Pandas 教程的“问题 → 说明 → 完整代码 → 结果解读”形式。
本教程可独立运行，不依赖之前 Notebook 内存里的变量。两个模块末尾各有两道综合练习，
按“问题要求 → 空白 Python 代码格 → 折叠参考答案”排列。

两个主要模块：

1. **Matplotlib：** 认识图的对象，用显式的 `fig, ax` 控制图形、样式、坐标、子图与导出。
2. **Pandas 绘图：** 从 Series / DataFrame 快速画图，再用 Matplotlib 调整细节。

最后用 Plotly 实际演示一张可悬停、缩放、切换曲线的交互图，再简要介绍 hvPlot 与 Altair。
依赖声明已包含 Plotly；所有代码可离线运行。hvPlot 与 Altair 只介绍写法，不作为必装依赖。

**数据：** 使用课程唯一真实源 `tutorials/data/posts_two_weeks.parquet`，
筛选四川两周的可用帖子。图中的指标描述采集记录或已观测用户，不代表全平台覆盖。
城市标签沿用源字段；不从匿名用户 ID、文字或位置推断个人身份。
输出图片写到 `data/output/plotting_tutorial/`。

### 目录与讲解安排

- [开场：选图与准备统计表](#setup)
- [模块一：使用 Matplotlib](#matplotlib)
  - [A1．Figure、Axes、Axis 与两种接口](#a1)
  - [A2．折线图：时间、样式、图例与注释](#a2)
  - [A3．柱形图：比较类别数量](#a3)
  - [A4．直方图：观察分布](#a4)
  - [A5．散点图：观察两个指标的关系](#a5)
  - [A6．子图、比例格式与坐标尺度](#a6)
  - [A7．热力图、色条与保存图片](#a7)
  - [模块一综合练习（两题）](#module1-exercises)
- [模块二：使用 Pandas 绘图](#pandas)
  - [B1．Series.plot 与 DataFrame.plot](#b1)
  - [B2．柱形图、直方图与箱线图](#b2)
  - [B3．散点图、六边形分箱与堆叠图](#b3)
  - [B4．子图与 Matplotlib 配合](#b4)
  - [模块二综合练习（两题）](#module2-exercises)
- [拓展：Plotly 交互图演示，hvPlot 与 Altair](#interactive)
- [速查与参考来源](#reference)

**约 45–50 分钟的基础路线：** 开场 5 分钟 → A1–A2 10 分钟 → A3–A5 10 分钟 →
A6 5 分钟 → B1–B2 8 分钟 → B4 4 分钟 → Plotly 交互演示 5 分钟。
A7 与 B3 可在另一节课继续讲。全部图形示例都保留在同一本 Notebook 中。
上述时间不含综合练习；每题建议 10–15 分钟，可课上选做或课后完成。

<a id="setup"></a>
## 开场：先决定表达什么，再决定用什么图

| 问题 | 适合的图 | 横轴与纵轴 |
|---|---|---|
| 记录数量怎样随日期变化？ | 折线图 | 日期、记录数 |
| 哪些城市记录更多？ | 柱形图 | 类别、记录数 |
| 文本长度主要落在哪些范围？ | 直方图 | 长度区间、区间内帖子数 |
| 用户帖子数与活跃天数有什么关系？ | 散点图 | 发帖数、观测日期数 |
| 不同组的文本长度分布有何差异？ | 箱线图 | 分组、中位数与分位范围 |
| 星期与小时的记录强度怎样组合？ | 热力图 | 小时、星期，颜色表示强度 |

同一张表可以画许多图，图形选择取决于问题。柱形图按类别比较，直方图按数值区间分箱，
二者虽然都画柱子，但横轴含义不同。画图前还要明确一行是帖子、用户还是日期。

In [ ]:
from pathlib import Path
from zoneinfo import ZoneInfo
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from matplotlib.ticker import PercentFormatter
from IPython.display import display

assert (Path(sys.prefix) / 'conda-meta').exists(), '请在右上角选择 Conda 内核'
# Notebook目录下用data/，项目根目录下用tutorials/data/。
TUTORIAL_DIR = Path('.') if Path('data/posts_two_weeks.parquet').is_file() else Path('tutorials')
DATA = TUTORIAL_DIR / 'data/posts_two_weeks.parquet'
assert DATA.is_file(), '请从 tutorials 或项目根目录启动内核，并保留 tutorials/data'
ROOT = TUTORIAL_DIR.resolve().parent
OUT = ROOT / 'data/output/plotting_tutorial'
OUT.mkdir(parents=True, exist_ok=True)
available = {font.name for font in font_manager.fontManager.ttflist}
CJK_FONT = next((name for name in ['Noto Sans CJK SC', 'Arial Unicode MS',
               'Microsoft YaHei', 'SimHei', 'Songti SC', 'Heiti TC', 'Noto Sans CJK JP']
               if name in available), None)
if CJK_FONT:
    plt.rcParams['font.family'] = CJK_FONT
plt.rcParams.update({'font.size': 11, 'figure.figsize': (9, 4),
                     'axes.unicode_minus': False, 'axes.spines.top': False,
                     'axes.spines.right': False})
pd.set_option('display.max_rows', 8)
pd.set_option('display.max_columns', 7)
pd.set_option('plotting.backend', 'matplotlib')

# 无中文字体的电脑使用英文图题，避免缺字；正文仍为中文。
def label(chinese, english):
    return chinese if CJK_FONT else english

print('Pandas：', pd.__version__, 'Matplotlib：', matplotlib.__version__)
print('当前 Python：', sys.executable)
print('中文字体：', CJK_FONT or '未找到；图中文字使用英文回退')

### 准备统计表：本教程所有图共用这些数据

先读取帖子表，再生成日期表、用户表和城市计数。
时间统一转为上海时区，再提取日期、小时和星期。`has_work` 只是“工作”一词的字面命中，
它不等于工作状态、职业或情绪分类。

In [ ]:
df = pd.read_parquet(DATA,
    columns=['post_id', 'user_id', 'created_at', 'city', 'text'],
    filters=[('province_code', '==', '51'), ('usable_text', '==', True)])
df['created_at'] = pd.to_datetime(df['created_at'], utc=True).dt.tz_convert('Asia/Shanghai')
df = df.assign(date=df['created_at'].dt.normalize(),
               hour=df['created_at'].dt.hour,
               weekday=df['created_at'].dt.dayofweek,
               text_length=df['text'].str.len(),
               has_work=df['text'].str.contains('工作', regex=False, na=False))
calendar = pd.date_range('2022-08-12', '2022-08-25', tz='Asia/Shanghai', freq='D')
daily = df.groupby('date').agg(n_posts=('post_id', 'size'),
                              n_users=('user_id', 'nunique'),
                              work_share=('has_work', 'mean')).reindex(calendar)
daily.index.name = 'date'
daily[['n_posts', 'n_users']] = daily[['n_posts', 'n_users']].fillna(0).astype('int64')
users = df.groupby('user_id').agg(n_posts=('post_id', 'size'),
                                active_days=('date', 'nunique'),
                                mean_text_length=('text_length', 'mean'))
city_counts = df['city'].value_counts()
print('四川记录数、用户数、日历天数：', len(df), len(users), len(calendar))
display(daily.head(3))
display(users.head(3))

**怎么看结果：** `daily` 每行是一天，`users` 每行是一个匿名用户，原始 `df` 每行是一条帖子。
一位用户可以出现在多个日期，所以每日用户数相加通常大于窗口中的不同用户数。
日期没有采集记录时，计数可以是 0，但比例没有分母，应保持缺失。

<a id="matplotlib"></a>
# 模块一：使用 Matplotlib

<a id="a1"></a>
## A1．Figure、Axes、Axis 与两种接口

- **Figure（`fig`）：** 整张图，可包含一个或多个绘图区；负责整体标题、布局与保存。
- **Axes（`ax`）：** 一个绘图区，负责其中的线、点、图题与轴标签。单个对象也叫 Axes。
- **Axis（`ax.xaxis` / `ax.yaxis`）：** 一个坐标轴的对象，管理刻度与格式。
- **Artist：** 图上可绘制对象的统称，如线条、文字、矩形。刚入门不需要记住各类名称。

`fig, ax = plt.subplots()` 同时创建整张图与一个绘图区。
`plt.plot(...)` 使用当前绘图区，适合短演示；`ax.plot(...)` 明确指定画到哪里，
便于做子图和复用代码。本教程主要使用后一种写法。
以下小数列专门展示接口，**不是研究数据**。

In [ ]:
x_demo = [1, 2, 3, 4]
y_demo = [1, 4, 2, 3]
plt.figure(figsize=(6, 3), layout='constrained')
plt.plot(x_demo, y_demo, marker='o')
plt.xlabel('x')
plt.ylabel('y')
plt.title(label('pyplot：操作当前绘图区', 'pyplot: current Axes'))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), layout='constrained')
line, = ax.plot(x_demo, y_demo, marker='o')
ax.set(xlabel='x', ylabel='y', title=label('ax：操作指定绘图区', 'ax: explicit Axes'))
print('对象类型：', type(fig).__name__, type(ax).__name__, type(line).__name__)
plt.show()

`ax.plot(...)` 返回线对象的列表；`line, = ...` 在只有一条线时取出该对象。
两格画出相同数据，但操作对象的方式不同。`figsize=(6, 3)` 的单位是英寸，
`layout='constrained'` 自动为标题和标签安排空间。

<a id="a2"></a>
## A2．折线图：时间、样式、图例与注释

折线图适合有明确顺序的横轴，特别是时间。
`ax.plot(x, y)` 的 x、y 要一一对应；只有 y 时，横轴默认是 0、1、2 等位置。
`marker` 设置点形状，`linewidth` 设置线宽，`color` 设置颜色。
`label` 给图例命名，之后调用 `legend()` 显示它。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4), layout='constrained')
ax.plot(daily.index, daily['n_posts'].to_numpy(), marker='o', linewidth=2,
        color='#2563a6', label=label('每日采集记录', 'Collected posts per day'))
ax.set(title=label('两周内每天采集了多少条记录？', 'Collected posts over two weeks'),
       xlabel=label('上海时区日期', 'Date (Asia/Shanghai)'),
       ylabel=label('记录数', 'Collected posts'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2, tz=ZoneInfo('Asia/Shanghai')))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d', tz=ZoneInfo('Asia/Shanghai')))
ax.grid(axis='y', alpha=0.25)
ax.legend(loc='upper left')
peak_date = daily['n_posts'].idxmax()
peak_value = int(daily.loc[peak_date, 'n_posts'])
ax.annotate(label('窗口内最高记录数', 'Maximum observed count'),
    xy=(peak_date, peak_value), xytext=(-100, -42), textcoords='offset points',
    arrowprops={'arrowstyle': '->', 'color': '#475569'}, fontsize=10)
plt.show()

**怎么看结果：** 点对应日期，线连接相邻日期。最高点仅是这个采集窗口内的最高记录数，
不能单凭波峰解释事件原因。`annotate` 的 `xy` 是被指向的数据位置，
`xytext` 配合 `textcoords='offset points'` 表示文字相对该点的排版偏移。

<a id="a3"></a>
## A3．柱形图：比较类别数量

城市类别名称较长，用水平柱形图 `barh` 比竖直柱形图更易读。
先按记录数选出前八项，再从小到大排列，使较长的柱位于上方。
柱形图以长度表达数量，数量图从 0 开始有助于保持比较准确。

In [ ]:
top_cities = city_counts.head(8).sort_values()
fig, ax = plt.subplots(figsize=(10, 4.4), layout='constrained')
bars = ax.barh(top_cities.index.astype(str), top_cities.to_numpy(dtype=float), color='#2563a6')
ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=10)
ax.set(title=label('采集记录最多的八个来源城市标签', 'Top eight source city labels'),
       xlabel=label('记录数', 'Collected posts'), ylabel=label('城市标签', 'City label'))
ax.set_xlim(0, float(top_cities.max()) * 1.18)
ax.grid(axis='x', alpha=0.2)
plt.show()

图只展示前八项，没有把它们当作全部城市。城市名沿用源字段，如 `Chengdu`。
数量受城市规模与采集覆盖影响，不能从这张图直接比较居民使用倾向。

<a id="a4"></a>
## A4．直方图：观察分布

直方图将数值划入区间（bins），柱高表示区间里的观测数量。
以下每个观测是一条帖子的字符数。`bins` 改变细节呈现，但没有改变输入数据。
为了聚焦主体分布，两个图都只展示不超过 95% 分位数的正文长度，并明确报告保留范围。

In [ ]:
lengths = df['text_length'].dropna().to_numpy(dtype=float)
length_limit = float(np.quantile(lengths, 0.95))
visible_lengths = lengths[lengths <= length_limit]
print('完整记录数：', len(lengths), '图示记录数：', len(visible_lengths),
      '长度上界：', length_limit)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True, layout='constrained')
for ax, bins in zip(axes, [10, 30]):
    ax.hist(visible_lengths, bins=bins, color='#2563a6', edgecolor='white')
    ax.set(title=f'bins={bins}', xlabel=label('正文字符数', 'Text characters'))
    ax.grid(axis='y', alpha=0.2)
axes[0].set_ylabel(label('区间内帖子数', 'Posts in each bin'))
fig.suptitle(label('相同数据，不同分箱数（长度 ≤ 95% 分位数）',
                   'Same data, different bins (length <= 95th percentile)'))
plt.show()

**怎么看结果：** 分箱越多，细节越多，也更容易显得起伏。
中文字符数不等于词数；上图没有展示长文本尾部，完整统计仍使用原始帖子表。
`density=True` 会改为概率密度，此时各柱**面积**的总和为 1，柱高不再表示帖子数量。

<a id="a5"></a>
## A5．散点图：观察两个指标的关系

散点图的每个点是一行数据，两个轴对应两个数值指标。
这里每个点是一位用户；横轴为窗口发帖数，纵轴为观测到发帖的日期数。
`s` 是点的面积，`alpha` 是透明度。透明度可缓解遮挡，但不能完全解决重复点重叠。
本例使用全部已观测用户，不抽样。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
ax.scatter(users['n_posts'].to_numpy(dtype=float), users['active_days'].to_numpy(dtype=float),
           s=16, alpha=0.15, color='#2563a6', edgecolors='none')
ax.set(title=label('同一用户的帖子数与观测日期数', 'Posts and observed days per user'),
       xlabel=label('窗口发帖数', 'Posts in window'),
       ylabel=label('观测到发帖的日期数', 'Observed posting days'))
ax.set_yticks(range(1, 15))
ax.grid(alpha=0.2)
plt.show()

**怎么看结果：** 这些指标都是整数，很多用户会落在同一个位置。
发帖数越多，可能覆盖更多日期，但图不能证明因果关系，也没有反映没有被采集到的使用行为。
若为演示做抽样，需标明抽样数和方法；这里不需要抽样。

<a id="a6"></a>
## A6．子图、比例格式与坐标尺度

### 同一画布里比较数量与比例

`plt.subplots(2, 1, sharex=True)` 创建上下两个绘图区，共用横轴。
数量和比例单位不同，分别画在各自的纵轴上。
`PercentFormatter(xmax=1)` 将 0.05 显示成 5%，没有改变底层数据。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True, layout='constrained')
axes[0].plot(daily.index, daily['n_posts'].to_numpy(), marker='o', color='#2563a6')
axes[1].plot(daily.index, daily['work_share'].to_numpy(dtype=float), marker='o', color='#c56a28')
axes[0].set_ylabel(label('采集记录数', 'Collected posts'))
axes[1].set_ylabel(label('工作词命中比例', 'Work keyword share'))
axes[1].set_xlabel(label('上海时区日期', 'Date (Asia/Shanghai)'))
axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d', tz=ZoneInfo('Asia/Shanghai')))
for ax in axes:
    ax.grid(axis='y', alpha=0.2)
fig.suptitle(label('每日数量与词命中比例', 'Daily counts and keyword proportions'))
plt.show()

比例的分母是当日全部采集记录，既不是居民数，也不是当日独立用户数。
上下两图共用时间轴，便于看同日变化；没有用双纵轴叠加两种指标。

### 长尾指标：线性与对数坐标

先按窗口发帖数从低到高排列用户，用排名看指标的跨度。
`set_yscale('log')` 只改变纵轴显示尺度，不是对原表赋新值。
对数轴只能显示正值；这里每个已观测用户至少有一条记录。

In [ ]:
ranked_posts = users['n_posts'].sort_values().to_numpy(dtype=float)
ranks = np.arange(1, len(ranked_posts) + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, layout='constrained')
for ax in axes:
    ax.plot(ranks, ranked_posts, color='#2563a6')
    ax.set_xlabel(label('按发帖数升序排列的用户排名', 'User rank by ascending post count'))
    ax.set_ylabel(label('窗口发帖数', 'Posts in window'))
    ax.grid(alpha=0.2)
axes[0].set_title(label('线性纵轴', 'Linear y-axis'))
axes[1].set_title(label('对数纵轴', 'Log y-axis'))
axes[1].set_yscale('log')
plt.show()

**怎么看结果：** 横轴是排序后的用户排名，不是时间，所以这不是用户发帖随时间变化的曲线。
对数轴上相同间距表示相同倍数。看图时必须说明所用尺度。

<a id="a7"></a>
## A7．热力图、色条与保存图片

热力图把矩阵的数值映射为颜色。这里按“星期 × 小时”统计帖子数，
再除以窗口中各星期出现的日历天数，得到“每个对应日历日、该小时的平均采集记录数”。
横轴小时按上海时区解释；颜色单位与矩阵单位相同。

In [ ]:
weekday_hour = pd.crosstab(df['weekday'], df['hour']).reindex(
    index=range(7), columns=range(24), fill_value=0)
weekday_days = pd.Series(calendar.dayofweek).value_counts().reindex(range(7), fill_value=0)
weekday_hour_mean = weekday_hour.div(weekday_days.where(weekday_days.gt(0)), axis=0)
weekday_labels = ([label('周一', 'Mon'), label('周二', 'Tue'), label('周三', 'Wed'),
                   label('周四', 'Thu'), label('周五', 'Fri'), label('周六', 'Sat'), label('周日', 'Sun')])
fig, ax = plt.subplots(figsize=(10, 3.8), layout='constrained')
image = ax.imshow(weekday_hour_mean.to_numpy(dtype=float), aspect='auto',
                  cmap='viridis', origin='upper')
ax.set_xticks(range(0, 24, 3), labels=[str(h) for h in range(0, 24, 3)])
ax.set_yticks(range(7), labels=weekday_labels)
ax.set(title=label('星期与小时的平均采集强度', 'Average collected posts by weekday and hour'),
       xlabel=label('上海时区小时', 'Hour (Asia/Shanghai)'),
       ylabel=label('星期', 'Weekday'))
fig.colorbar(image, ax=ax, label=label('对应日历日每小时平均记录数', 'Mean posts per hour per matching day'))
heatmap_fig = fig
# 在 show 前保存到独立输出目录；PNG 便于插入 PPT，SVG 可保留矢量文字与图形。
heatmap_fig.savefig(OUT / 'weekday_hour_heatmap.png', dpi=180, bbox_inches='tight')
heatmap_fig.savefig(OUT / 'weekday_hour_heatmap.svg', bbox_inches='tight')
plt.show()
print('输出目录：', OUT.relative_to(ROOT))

`imshow` 适合规则矩阵；`fig.colorbar(image, ax=ax)` 解释颜色与数值的关系。
顺序色图 `viridis` 适合从低到高的强度。相对零点有正负差异时，再考虑发散色图。
此图只覆盖两周，颜色深浅不是某种作息或出行规律的证明。

PNG 是像素图，`dpi` 影响导出像素密度；SVG 是矢量格式。Matplotlib 默认将 SVG 文字转为路径，
可以减少跨电脑的字体依赖；若改成保存可编辑文字，则需要核对目标电脑字体。保存使用 `fig.savefig`，
保留对象引用并在 `show` 前导出，可避免误保存空白的当前画布。

<a id="module1-exercises"></a>
## 模块一综合练习

每题建议 10–15 分钟，可课上选做或课后完成。先在空白代码格作答，再展开参考答案。解读可写在代码格的注释或 `print` 中；答案代码可复制到空白格运行。

### 综合练习 1.1：每日记录数与关键词比例：双子图

使用开场准备的 `daily`，用 Matplotlib 的 `fig, axes` 完成：

1. 建立上下两个子图，共享日期横轴。上图绘制每日帖子记录数，下图绘制正文含“工作”的记录比例。
2. 两条线分别设置颜色、点标记和图例；添加图题、单位和浅色网格。比例轴使用百分比刻度。
3. 在上图标注记录数最多的一天；若并列，标注最早的一天即可。设置简洁的日期刻度。
4. 在 `plt.show()` 前保存为 `OUT / 'exercise_m1_daily.png'`，设置 `dpi=150`。
5. 用 `print` 写一句解读，说明比例的分子、分母，以及为什么它不表示用户的工作状态。

可回看 A2、A6、A7；`idxmax()` 可找最大值的位置，`PercentFormatter(xmax=1)` 可格式化 0–1 比例。
没有记录的日期，比例应保持缺失，不要填为 0。

<details><summary>展开参考答案与解读（建议先完成练习）</summary><p>参考思路：数量与比例的单位不同，放在共享日期的两个子图中更容易读取。 上图统计帖子记录，下图分子是当天正文含“工作”的记录数，分母是当天全部采集记录数。</p><p>这只是字面关键词比例。正文可能是引用、广告或转述，一位用户也可以发多条帖； 不能据此判断用户是否在工作。最高点只描述本采集窗口，不代表全平台高峰。</p><pre><code class="language-python">m1e1_fig, m1e1_axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, layout=&#x27;constrained&#x27;)
m1e1_axes[0].plot(daily.index, daily[&#x27;n_posts&#x27;], color=&#x27;#2878B5&#x27;, marker=&#x27;o&#x27;,
                 label=label(&#x27;帖子记录数&#x27;, &#x27;Post records&#x27;))
m1e1_peak = daily[&#x27;n_posts&#x27;].idxmax()
m1e1_axes[0].annotate(
    label(&#x27;窗口内最高点&#x27;, &#x27;Window maximum&#x27;),
    xy=(m1e1_peak, daily.loc[m1e1_peak, &#x27;n_posts&#x27;]), xytext=(12, -30),
    textcoords=&#x27;offset points&#x27;, arrowprops={&#x27;arrowstyle&#x27;: &#x27;-&gt;&#x27;},
)
m1e1_axes[0].set(title=label(&#x27;每日记录数&#x27;, &#x27;Daily post records&#x27;),
               ylabel=label(&#x27;帖子记录数（条）&#x27;, &#x27;Post records&#x27;))
m1e1_axes[0].set_ylim(bottom=0)
m1e1_axes[1].plot(daily.index, daily[&#x27;work_share&#x27;], color=&#x27;#E07A35&#x27;, marker=&#x27;s&#x27;,
                 label=label(&#x27;“工作”关键词比例&#x27;, &#x27;Work-keyword share&#x27;))
m1e1_axes[1].set(title=label(&#x27;每日关键词比例&#x27;, &#x27;Daily keyword share&#x27;),
               xlabel=label(&#x27;上海时区日期&#x27;, &#x27;Date (Asia/Shanghai)&#x27;),
               ylabel=label(&#x27;占当天记录的比例&#x27;, &#x27;Share of daily records&#x27;), ylim=(0, None))
m1e1_axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=1))
m1e1_locator = mdates.AutoDateLocator(tz=ZoneInfo(&#x27;Asia/Shanghai&#x27;))
m1e1_axes[1].xaxis.set_major_locator(m1e1_locator)
m1e1_axes[1].xaxis.set_major_formatter(
    mdates.ConciseDateFormatter(m1e1_locator, tz=ZoneInfo(&#x27;Asia/Shanghai&#x27;)))
for m1e1_ax in m1e1_axes:
    m1e1_ax.legend(loc=&#x27;upper left&#x27;)
    m1e1_ax.grid(axis=&#x27;y&#x27;, alpha=0.25)
m1e1_fig.savefig(OUT / &#x27;exercise_m1_daily.png&#x27;, dpi=150, bbox_inches=&#x27;tight&#x27;)
plt.show()
print(&#x27;比例 = 当天含“工作”的帖子记录数 / 当天全部采集记录数；关键词命中不代表用户正在工作。&#x27;)</code></pre></details>

### 综合练习 1.2：城市数量与文本长度：类别比较和分布

使用 `city_counts` 与 `df['text_length']`，仅用 Matplotlib 绘图：

1. 建立左右两个子图。左图显示记录数最多的 6 个城市，用横向柱形图，数量最大的城市放在最上方。
   在柱末标出数量，并给数值轴留出空间，避免文字超出图框。
2. 右图绘制文本长度直方图，取非缺失长度中不超过第 95 百分位的记录，使用 20 个箱。
   在图题或图注中写明截取范围，并打印阈值及保留记录数。
3. 每张子图写出图题、坐标含义与单位；添加浅色网格并调整布局。
4. 用 `print` 说明两图中柱子的横轴含义有什么不同，以及直方图中较长文本去了哪里。

可回看 A3–A4、A6。`ax.bar_label` 可标注柱子，`np.quantile(values, 0.95)` 可计算阈值。
这里截取长度只是为了显示主体分布，不要修改原始 `df`。

<details><summary>展开参考答案与解读（建议先完成练习）</summary><p>参考思路：先按记录数选择前六个城市，再升序排列，使最大的横向柱子出现在最上方。 城市柱形图按类别比较数量；直方图按连续数值区间统计频数，两者不能交换解释。</p><p>超过阈值的文本只从这张直方图的显示范围排除，仍保留在原表中。 因为长度是整数，阈值处可能有并列，保留比例未必恰好为 95%；这张图也不能描述全部长尾。</p><pre><code class="language-python">m1e2_top = city_counts.head(6).sort_values()
m1e2_lengths = df[&#x27;text_length&#x27;].dropna().to_numpy(dtype=float)
m1e2_limit = float(np.quantile(m1e2_lengths, 0.95))
m1e2_visible = m1e2_lengths[m1e2_lengths &lt;= m1e2_limit]
m1e2_fig, m1e2_axes = plt.subplots(1, 2, figsize=(12, 4.5), layout=&#x27;constrained&#x27;)
m1e2_bars = m1e2_axes[0].barh(m1e2_top.index, m1e2_top.values, color=&#x27;#2878B5&#x27;)
m1e2_axes[0].bar_label(m1e2_bars, fmt=&#x27;%d&#x27;, padding=3, fontsize=9)
m1e2_axes[0].set(title=label(&#x27;记录数最多的六个城市&#x27;, &#x27;Six cities with most records&#x27;),
               xlabel=label(&#x27;帖子记录数（条）&#x27;, &#x27;Post records&#x27;),
               ylabel=label(&#x27;源数据城市标签&#x27;, &#x27;Source city label&#x27;),
               xlim=(0, float(m1e2_top.max()) * 1.22))
m1e2_axes[0].grid(axis=&#x27;x&#x27;, alpha=0.25)
m1e2_axes[1].hist(m1e2_visible, bins=20, color=&#x27;#E07A35&#x27;, edgecolor=&#x27;white&#x27;)
m1e2_axes[1].set(
    title=label(f&#x27;文本长度 ≤ 第95百分位（{m1e2_limit:.0f}字符）&#x27;,
                f&#x27;Text length &lt;= P95 ({m1e2_limit:.0f} characters)&#x27;),
    xlabel=label(&#x27;正文长度（字符）&#x27;, &#x27;Text length (characters)&#x27;),
    ylabel=label(&#x27;区间内帖子记录数（条）&#x27;, &#x27;Post records per bin&#x27;))
m1e2_axes[1].grid(axis=&#x27;y&#x27;, alpha=0.25)
plt.show()
print(f&#x27;第95百分位阈值：{m1e2_limit:g}字符；保留 {len(m1e2_visible):,} / {len(m1e2_lengths):,} 条。&#x27;)
print(&#x27;左图横轴是记录数，城市类别在纵轴；右图横轴是长度区间，柱高是该区间记录数。&#x27;)
print(&#x27;更长的文本只在这张直方图中未显示，原始数据仍保留；不能用此图描述完整长尾。&#x27;)</code></pre></details>

<a id="pandas"></a>
# 模块二：使用 Pandas 绘图

<a id="b1"></a>
## B1．Series.plot 与 DataFrame.plot

在默认 Matplotlib 后端下，Pandas `.plot()` 帮我们从表格取数并调用绘图，通常返回 Axes。
因此返回的 `ax` 仍能使用 `set_title`、`grid` 等 Matplotlib 方法。
本教程开场明确设置了 `plotting.backend='matplotlib'`。

### 一列数值：Series.plot

默认画折线图，索引作为横轴。这里 `daily` 的索引已经是按日排列的日期。

In [ ]:
ax = daily['n_posts'].plot(figsize=(9, 3.5), marker='o', color='#2563a6',
                            title=label('Pandas：每日记录数', 'Pandas: daily collected posts'))
ax.set_xlabel(label('日期', 'Date'))
ax.set_ylabel(label('记录数', 'Collected posts'))
ax.grid(axis='y', alpha=0.2)
plt.show()

### 多列数值：DataFrame.plot

默认将选中的每个数值列画成一条线，并根据列名生成图例。
可以用 `y=` 明确指定列；若要用普通列作横轴，使用 `x=`。
本例把每日帖子数与每日独立用户数放在一起，两者都按“个数”显示，但统计单位不同。

In [ ]:
counts_for_plot = daily[['n_posts', 'n_users']].rename(columns={
    'n_posts': label('帖子记录数', 'Posts'), 'n_users': label('独立用户数', 'Distinct users')})
ax = counts_for_plot.plot(marker='o', figsize=(9, 3.5))
ax.set(title=label('同一天：帖子数与独立用户数', 'Daily posts and distinct users'),
       xlabel=label('日期', 'Date'), ylabel=label('数量', 'Count'))
ax.grid(axis='y', alpha=0.2)
plt.show()

**怎么看结果：** 一位用户一天可发多条，所以每日记录数不小于每日独立用户数。
这两条线仍然来自同一张帖子表；独立用户数是一种按日去重的统计。
日期应在画线前排序，Pandas 不会替我们判断不规则类别是否适合连成线。

<a id="b2"></a>
## B2．柱形图、直方图与箱线图

### 改变图型：kind 或 plot 的具体方法

`series.plot(kind='barh')` 与 `series.plot.barh()` 都能画水平柱形图。
后一种写法在 Notebook 的自动补全中更容易发现。
`ax=` 可以让 Pandas 将图画到我们预先创建的绘图区里。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.3), layout='constrained')
top_cities.plot.barh(ax=ax, color='#2563a6', legend=False)
ax.set(title=label('Pandas：城市记录数', 'Pandas: collected posts by city'),
       xlabel=label('记录数', 'Collected posts'), ylabel=label('城市标签', 'City label'))
ax.grid(axis='x', alpha=0.2)
plt.show()

### plot.hist：画连续或离散数值的分布

延续 A4 的相同显示范围，不另换数据。输入一列数值，让绘图库负责分箱。
`DataFrame.hist()` 也能画直方图，但接口与 `DataFrame.plot.hist()` 并不完全相同。

In [ ]:
length_for_plot = pd.Series(visible_lengths, name=label('正文字符数', 'Text characters'))
ax = length_for_plot.plot.hist(bins=20, edgecolor='white', color='#2563a6', figsize=(8, 3.5))
ax.set(title=label('Pandas：正文长度分布（≤ 95% 分位数）',
                   'Pandas: text length distribution (<= 95th percentile)'),
       xlabel=label('正文字符数', 'Text characters'), ylabel=label('帖子数', 'Posts'))
plt.show()

### 箱线图：中位数、四分位数与组间分布

把帖子分成工作日与周末两组，比较文本长度分布。
箱体通常跨第 1 到第 3 四分位数；中间的线是中位数；默认须线伸到 1.5 倍四分位距以内的最远点。
须线以外的点叫离群点，它们不一定是数据错误。
为了看清箱体，本例隐藏离群点标记，并把 y 轴显示范围限制在 A4 的 95% 分位数上界。
两种显示设置都不会删除统计输入，箱体与须线仍使用全部长度计算。

In [ ]:
weekend = df['weekday'].ge(5)
# reset_index 让两组独立排列；列长度不同的尾部缺失由箱线图忽略，不补为 0。
box_data = pd.DataFrame({
    label('工作日', 'Weekdays'): df.loc[~weekend, 'text_length'].reset_index(drop=True),
    label('周末', 'Weekends'): df.loc[weekend, 'text_length'].reset_index(drop=True),
})
display(box_data.describe().loc[['count', '25%', '50%', '75%']])
ax = box_data.plot.box(figsize=(7, 3.7), showfliers=False)
ax.set_ylim(0, length_limit)
ax.set(title=label('工作日与周末正文长度：局部显示', 'Text length by day type: limited y range'),
       ylabel=label('正文字符数', 'Text characters'))
ax.grid(axis='y', alpha=0.2)
plt.show()

**怎么看结果：** 图比较的是帖子等权的长度分布，而非每位用户等权的长度分布。
图中局部显示不改变中位数与四分位数的计算。颜色或箱体差异不能替代统计推断。

<a id="b3"></a>
## B3．散点图、六边形分箱与堆叠图

### plot.scatter：指定 x、y 列

`x` 与 `y` 写列名；Pandas 负责从 DataFrame 取出数值。
使用与 A5 相同的全部用户，便于对照两种接口。

In [ ]:
ax = users.plot.scatter(x='n_posts', y='active_days', s=16, alpha=0.15,
                         color='#2563a6', figsize=(8, 4))
ax.set(title=label('Pandas：用户发帖数与观测日期数', 'Pandas: posts and observed days per user'),
       xlabel=label('窗口发帖数', 'Posts in window'),
       ylabel=label('观测到发帖的日期数', 'Observed posting days'))
ax.set_yticks(range(1, 15))
plt.show()

### plot.hexbin：用格内计数缓解点重叠

`hexbin` 将散点落入六边形格子，默认以颜色表示格子内的点数。
这里每个点是一位用户，色条表示用户数，不是帖子数。
`gridsize` 决定分箱细度；`mincnt=1` 不着色没有点的格子。

In [ ]:
ax = users.plot.hexbin(x='n_posts', y='mean_text_length', gridsize=22,
                        mincnt=1, cmap='viridis', figsize=(8, 4))
ax.set(title=label('用户指标的六边形分箱', 'Hexagonal bins of user metrics'),
       xlabel=label('窗口发帖数', 'Posts in window'),
       ylabel=label('用户平均正文字符数', 'Mean text characters per user'))
ax.figure.axes[-1].set_ylabel(label('格内用户数', 'Users in bin'))
plt.show()

### 堆叠柱形图和面积图：只堆叠可以相加的互斥部分

将来源城市分成记录最多的三座城市与“其余城市”，每条帖子只属于其中一组。
这样同一天各列之和才等于当日全部帖子数。
若是“工作、饮食、旅行”等可同时命中的关键词标签，类别重叠，就不能直接解释为总量的分割。

In [ ]:
top3 = city_counts.head(3).index.tolist()
other_label = label('其余城市', 'Other cities')
city_groups = df['city'].where(df['city'].isin(top3), other_label)
daily_city = pd.crosstab(df['date'], city_groups).reindex(
    index=calendar, columns=top3 + [other_label], fill_value=0)
display(daily_city.head(3))
fig, axes = plt.subplots(2, 1, figsize=(10, 7), layout='constrained')
daily_city.plot.bar(stacked=True, ax=axes[0], rot=0,
                     color=['#2563a6', '#c56a28', '#50896d', '#b0b8c0'])
axes[0].set_xticks(range(0, 14, 2), labels=calendar[::2].strftime('%m-%d'))
axes[0].set(title=label('按日分组的堆叠柱形图', 'Stacked bars by date'),
            xlabel='', ylabel=label('记录数', 'Collected posts'))
daily_city.plot.area(stacked=True, ax=axes[1],
                      color=['#2563a6', '#c56a28', '#50896d', '#b0b8c0'])
axes[1].set(title=label('相同统计表的堆叠面积图', 'Stacked area from the same table'),
            xlabel=label('日期', 'Date'), ylabel=label('记录数', 'Collected posts'))
for ax in axes:
    ax.legend(loc='upper left', fontsize=9, framealpha=0.85)
plt.show()

柱形图更直接地表示每日数量；面积图连接相邻日期，便于看总量与组成变化，
但不能把柱间填充理解成新增的连续观测。Pandas 面积图会把缺失数值按 0 处理，
因此这里先构造明确的每日计数表；对缺失比例或测量值，不能直接使用这种零填充解释。

<a id="b4"></a>
## B4．子图与 Matplotlib 配合

### subplots=True：自动给每个数值列一张小图

返回值通常是一组 Axes。下面把每日记录数与每日独立用户数放到独立子图，
避免不同量级让某条线的变化难以辨认。

In [ ]:
axes = counts_for_plot.plot(subplots=True, sharex=True, figsize=(9, 5),
                             marker='o', layout=(2, 1), legend=False)
for ax, column in zip(np.asarray(axes).ravel(), counts_for_plot.columns):
    ax.set_title(column)
    ax.set_ylabel(label('数量', 'Count'))
    ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

### 先用 Pandas 画，再用 Matplotlib 定制

Pandas 负责从统计表取数；Matplotlib 负责绘图区、比例格式、网格和保存。
`ax=` 是两者配合的关键。所有风格修改仍作用于同一个绘图区。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.7), layout='constrained')
daily['work_share'].plot(ax=ax, marker='o', color='#c56a28', legend=False)
ax.set(title=label('Pandas 取数 + Matplotlib 调整显示', 'Pandas data + Matplotlib formatting'),
       xlabel=label('日期', 'Date'), ylabel=label('工作词命中比例', 'Work keyword share'))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(axis='y', alpha=0.25)
fig.savefig(OUT / 'daily_work_share.png', dpi=180, bbox_inches='tight')
plt.show()

**怎么看结果：** 图仍然显示同一条每日词命中比例，只是用百分比标注纵轴。
从 Pandas 进入 Matplotlib 不需要重新计算统计结果。

<a id="module2-exercises"></a>
## 模块二综合练习

每题建议 10–15 分钟，可课上选做或课后完成。先在空白代码格作答，再展开参考答案。解读可写在代码格的注释或 `print` 中；答案代码可复制到空白格运行。

### 综合练习 2.1：工作日与周末：按日均值比较城市

使用 `df`、`calendar` 与 `city_counts`，比较记录数最多的 5 个城市：

1. 生成一个“城市 × 日期类型”表，分别统计工作日（星期一至五）与周末（星期六、日）的记录数。
   可用 `pd.crosstab(df['city'], df['weekday'].ge(5))`，列 `False`、`True` 分别表示两类。
2. 从完整的 `calendar` 计算两类日历天数，将各类总记录数除以对应天数，得到每个城市的日均记录数。
   日历天数不能用帖子行数或某城市有记录的日期数替代。
3. 用 DataFrame 的 `.plot.bar(ax=...)` 画分组柱形图。设置单位、图例、标题和城市标签角度，数量轴从 0 开始。
4. 显示日均表，打印两个分母，解释为什么直接比较两类总记录数可能误导。

可回看 B1–B2、B4；使用 `reindex(..., fill_value=0)` 保留没有记录的类别，
`table.div(denominators, axis='columns')` 可逐列除以相应分母。
本练习按星期区分日期，未识别法定节假日或调休。

<details><summary>展开参考答案与解读（建议先完成练习）</summary><p>参考思路：本窗口有 10 个星期一至五、4 个周末日。总量会受到两类天数差异的影响， 因此各城市应使用同一完整日历作为分母，比较“每个日历日的采集记录数”。</p><p>工作日、周末是两个互斥类别。日均值是采集记录的平均值，不是用户数、城市人口或用户发帖概率。 差异也可能来自采集覆盖变化，不能直接归因于周末行为。</p><pre><code class="language-python">m2e1_counts = pd.crosstab(df[&#x27;city&#x27;], df[&#x27;weekday&#x27;].ge(5)).reindex(
    index=city_counts.head(5).index, columns=[False, True], fill_value=0)
m2e1_days = pd.Series({False: int((calendar.dayofweek &lt; 5).sum()),
                      True: int((calendar.dayofweek &gt;= 5).sum())})
m2e1_means = m2e1_counts.div(m2e1_days, axis=&#x27;columns&#x27;)
m2e1_means.columns = [label(&#x27;工作日日均&#x27;, &#x27;Weekday daily mean&#x27;),
                      label(&#x27;周末日均&#x27;, &#x27;Weekend daily mean&#x27;)]
m2e1_fig, m2e1_ax = plt.subplots(figsize=(10, 4.5), layout=&#x27;constrained&#x27;)
m2e1_means.plot.bar(ax=m2e1_ax, color=[&#x27;#2878B5&#x27;, &#x27;#E07A35&#x27;], rot=25)
m2e1_ax.set(title=label(&#x27;前五城市：工作日与周末日均记录数&#x27;, &#x27;Top five cities: mean records per calendar day&#x27;),
            xlabel=label(&#x27;源数据城市标签&#x27;, &#x27;Source city label&#x27;),
            ylabel=label(&#x27;日均帖子记录数（条/日）&#x27;, &#x27;Post records per calendar day&#x27;), ylim=(0, None))
m2e1_ax.legend(title=label(&#x27;日期类型&#x27;, &#x27;Day type&#x27;))
m2e1_ax.grid(axis=&#x27;y&#x27;, alpha=0.25)
plt.show()
display(m2e1_means)
print(&#x27;分母：工作日&#x27;, m2e1_days[False], &#x27;天；周末&#x27;, m2e1_days[True], &#x27;天。&#x27;)
print(&#x27;两类日历天数不同，直接比较总记录数会混入天数差异；日均值仍只描述采集记录。&#x27;)</code></pre></details>

### 综合练习 2.2：每日城市构成：比例面积图与总量

使用 B3 中已经准备好的 `daily_city`（前三城市和“其他城市”四个互斥类别）：

1. 计算每日四类总量，再将每行除以该日总量，得到比例表。总量为 0 的日期比例保持缺失。
2. 建立上下两个子图：上图用 `.plot.area(stacked=True, ax=...)` 绘制四类比例，纵轴为 0–100%；
   下图用 Series 的 `.plot.line(ax=...)` 绘制每日总记录数，数量轴从 0 开始。
3. 设置图题、轴标签、图例与网格；两个子图使用同一日期范围，日期标签不要重叠。
   Pandas 默认面积图会把缺失值填为 0，所以若有无记录日期，应先说明该默认处理不适合解释比例。
   当前数据每天均有记录，可先核对这一点后绘图。
4. 保存为 `OUT / 'exercise_m2_composition.png'`。用 `print` 解释为什么比例图要同时配一张总量图，
   以及为什么“其他城市”不能直接删去后仍称为全部记录构成。

可回看 B3–B4；`sum(axis=1)` 可得到每日总量，`div(..., axis='index')` 可逐行归一化。
两图都用 Pandas 的绘图入口；样式和保存通过返回的 Matplotlib Axes / Figure 控制。

<details><summary>展开参考答案与解读（建议先完成练习）</summary><p>参考思路：四类覆盖当天全部记录，因此有记录日期的行比例和为 1。 比例图突出构成变化，但同一个比例可以对应不同记录总量；配总量图能显示两种信息。</p><p>删除“其他城市”后，如果重新归一化，得到的是前三城市内部构成； 如果不重新归一化，三个比例之和通常小于 1。二者都不再是四类完整构成。 当前窗口没有零记录日期；下方检查避免把未来数据的缺失比例默认绘为 0。</p><pre><code class="language-python">m2e2_totals = daily_city.sum(axis=1)
m2e2_shares = daily_city.div(m2e2_totals.where(m2e2_totals.gt(0)), axis=&#x27;index&#x27;)
if not m2e2_totals.gt(0).all():
    raise ValueError(&#x27;存在无记录日期：比例缺失。请先设计缺口显示，避免面积图默认填0。&#x27;)
m2e2_fig, m2e2_axes = plt.subplots(2, 1, figsize=(10, 6), layout=&#x27;constrained&#x27;)
m2e2_shares.plot.area(stacked=True, ax=m2e2_axes[0],
                      color=[&#x27;#2878B5&#x27;, &#x27;#E07A35&#x27;, &#x27;#55A868&#x27;, &#x27;#AAAAAA&#x27;])
m2e2_totals.plot.line(ax=m2e2_axes[1], color=&#x27;#444444&#x27;, marker=&#x27;o&#x27;, legend=False)
m2e2_axes[0].set(title=label(&#x27;每日记录的城市构成&#x27;, &#x27;Daily city composition of records&#x27;),
               ylabel=label(&#x27;占当天记录的比例&#x27;, &#x27;Share of daily records&#x27;), ylim=(0, 1))
m2e2_axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=1))
m2e2_axes[0].legend(title=label(&#x27;城市类别&#x27;, &#x27;City group&#x27;),
                    loc=&#x27;upper left&#x27;, bbox_to_anchor=(1.02, 1))
m2e2_axes[1].set(title=label(&#x27;每日总记录数&#x27;, &#x27;Daily total records&#x27;),
               ylabel=label(&#x27;帖子记录数（条）&#x27;, &#x27;Post records&#x27;), ylim=(0, None))
for m2e2_ax in m2e2_axes:
    m2e2_ax.set_xlabel(label(&#x27;上海时区日期&#x27;, &#x27;Date (Asia/Shanghai)&#x27;))
    m2e2_ax.grid(axis=&#x27;y&#x27;, alpha=0.25)
    m2e2_ax.tick_params(axis=&#x27;x&#x27;, labelrotation=20)
m2e2_fig.savefig(OUT / &#x27;exercise_m2_composition.png&#x27;, dpi=150, bbox_inches=&#x27;tight&#x27;)
plt.show()
print(&#x27;四类互斥且覆盖全部记录；有记录日期的比例和为1。比例相同不意味着总记录数相同。&#x27;)
print(&#x27;删掉“其他城市”后，重新归一化只描述前三城市内部构成，不能仍称为全部记录构成。&#x27;)</code></pre></details>

<a id="interactive"></a>
## 拓展：Plotly、hvPlot、Altair

交互图可提供悬停查看数值、缩放、平移或选择数据。
它们适合课堂探索和网页展示；交互性本身不会修正错误的分母、排序或数据解释。
Matplotlib 也有交互后端，但本课使用的 Notebook 内嵌输出是静态图。

| 工具 | 常见写法与特点 | 适合怎样继续学 |
|---|---|---|
| Plotly Express | `px.line(table, x=..., y=...)`，常用图型可快速生成悬停和缩放图 | 把已经整理好的统计表用于网页或交互展示 |
| hvPlot | `table.hvplot.line(...)`，接口接近 Pandas，默认常用 Bokeh 后端 | 熟悉 `.plot()` 后继续探索交互数据表 |
| Altair | `Chart(...).mark_line().encode(...)`，声明字段映射到轴、颜色等视觉通道 | 学习字段类型、编码与交互选择的组合 |

### C1．具体演示：同一天有多少帖子、多少独立用户？

用 B1 的同一张每日统计表，画两条可以交互的折线。
图包含 14 个日期、两种计数，不带用户 ID 或帖子正文；日期仍表示上海的日历日期。
数据从 Python 送给浏览器后，浏览器就能完成悬停、缩放和图例切换，不需要在每次操作时重跑 Python。

本例使用 `plotly.express` 的 `px.line`，从 DataFrame 指定横轴和数值列，写法接近前面的 Pandas 绘图。
`Figure` 是普通图形对象，本例不需要额外的 Widget 环境或服务器。

In [ ]:
import plotly
import plotly.express as px

interactive_daily = daily[['n_posts', 'n_users']].reset_index()
# 已经转为上海时区；去掉时区标记只为了让前端按相同的日历日期显示。
interactive_daily['date'] = interactive_daily['date'].dt.tz_localize(None)
interactive_daily = interactive_daily.rename(columns={
    'n_posts': '帖子记录数', 'n_users': '独立用户数'})
print('Plotly 版本：', plotly.__version__)
display(interactive_daily.head(3))

In [ ]:
plotly_fig = px.line(
    interactive_daily, x='date', y=['帖子记录数', '独立用户数'], markers=True,
    labels={'date': '上海日历日期', 'value': '数量', 'variable': '指标'},
    color_discrete_sequence=['#2563a6', '#c56a28'],
    template='plotly_white',
)
plotly_fig.update_layout(
    title={'text': '两周每日发帖记录与独立用户数', 'x': 0.02},
    hovermode='x unified', height=580,
    font={'family': 'Arial Unicode MS, Microsoft YaHei, sans-serif', 'size': 14},
    margin={'l': 65, 'r': 25, 't': 95, 'b': 55},
    legend={'title_text': '点击图例切换曲线', 'orientation': 'h',
            'yanchor': 'bottom', 'y': 1.02, 'xanchor': 'left', 'x': 0},
)
plotly_fig.update_xaxes(
    type='date', tickformat='%m-%d', hoverformat='%Y-%m-%d',
    rangeslider={'visible': True, 'thickness': 0.16},
)
plotly_fig.update_yaxes(title_text='数量', rangemode='tozero', fixedrange=False)
plotly_fig.update_traces(hovertemplate='%{y:,.0f}<extra></extra>')
plotly_config = {'displaylogo': False, 'responsive': True,
                 'toImageButtonOptions': {'filename': 'daily_counts', 'scale': 2}}

# VS Code 使用 Plotly MIME 渲染；经典 Notebook 使用随输出保存的内嵌 JS。
plotly_fig.show(renderer='plotly_mimetype+notebook', config=plotly_config)

**教师演示顺序（约 5 分钟）：**

1. 将鼠标放到某一天的折线上：悬停框同时显示日期与两条曲线的数值。
2. 点击图例中的“帖子记录数”：隐藏蓝色曲线，只看独立用户数；再点击恢复。
   双击一项图例可只保留这一项，再双击恢复。
3. 拖动底部缩略图两端的把手，缩小到几天的时间范围；拖动选中区域可平移。
4. 在主图拖出矩形框查看局部，用右上角工具栏切换缩放和平移；双击主图恢复范围。
5. 可以展示工具栏的相机按钮：它将当前视图下载为静态 PNG；交互图本身保留在 Notebook 输出中。

**怎么看结果：** 交互让我们更方便地读同一天的两种计数、查看局部变化，统计定义仍与 B1 相同。
每天的独立用户数不是窗口内用户数的固定比例，也不能将不同日期用户数相加当作窗口去重用户数。
拖动滑块只改变视图，没有重新采集或改变原始数据。

Plotly 还可画交互柱形图、散点图、热力图、地图或动画；这些图仍需要正确的表格结构和指标定义。
这节课用一张折线图展示它的基本交互方式即可。

在 VS Code 中运行图形单元后，直接在输出区演示。图形数据与内嵌渲染资源随输出保存到 `.ipynb`，
无需另外导出 HTML 或打开网页。
新电脑若缺 Plotly，请按更新后的课程环境声明安装，或在课程 Conda 环境中运行
`conda install -n user-behavior-class -c conda-forge "plotly>=5.24,<7"`。

[Plotly 图形创建](https://plotly.com/python/creating-and-updating-figures/)、
[时间滑块](https://plotly.com/python/range-slider/)、
[Notebook 图形显示](https://plotly.com/python/renderers/)。

### C2．另外两种交互绘图入口：hvPlot 与 Altair

以下写法只作简短介绍，保持在文字代码块中，不作为这次演示的安装或运行前提。
仍用上面的 14 行统计表，纵轴列名写成“帖子记录数”。

**hvPlot：** 导入 Pandas 接口后，DataFrame 才拥有 `.hvplot` 入口。
```python
import hvplot.pandas
chart = interactive_daily.hvplot.line(
    x='date', y='帖子记录数', xlabel='上海日历日期', ylabel='采集记录数',
    title='每日采集记录数', width=750, height=300)
display(chart)
```
[hvPlot 官方入门](https://hvplot.holoviz.org/en/docs/latest/tutorials/getting_started_pandas.html)

**Altair：** `T` 表示时间字段，`Q` 表示数量字段；`.interactive()` 添加轴缩放和平移。
```python
import altair as alt
chart = (alt.Chart(interactive_daily).mark_line(point=True).encode(
    x=alt.X('date:T', title='上海日历日期'),
    y=alt.Y('帖子记录数:Q', title='采集记录数'),
    tooltip=['date:T', '帖子记录数:Q'])
    .properties(title='每日采集记录数', width=650, height=280)
    .interactive())
display(chart)
```
[Altair 官方入门](https://altair-viz.github.io/getting_started/starting.html)

<a id="reference"></a>
## 速查：同一个目的，两种入口

| 目的 | Matplotlib | Pandas（默认 Matplotlib 后端） |
|---|---|---|
| 折线 | `ax.plot(x, y)` | `series.plot()`、`table.plot.line()` |
| 柱形 | `ax.bar(x, height)`、`ax.barh(y, width)` | `.plot.bar()`、`.plot.barh()` |
| 直方 | `ax.hist(values, bins=20)` | `.plot.hist(bins=20)` |
| 箱线 | `ax.boxplot(values)` | `.plot.box()`、`.boxplot()` |
| 散点 | `ax.scatter(x, y)` | `.plot.scatter(x='a', y='b')` |
| 六边形分箱 | `ax.hexbin(x, y)` | `.plot.hexbin(x='a', y='b')` |
| 面积 | `ax.fill_between(...)`、`ax.stackplot(...)` | `.plot.area(stacked=True)` |
| 子图 | `fig, axes = plt.subplots(...)` | `.plot(subplots=True)` 或传 `ax=` |
| 图题、图例、保存 | `ax.set_title(...)`、`ax.legend()`、`fig.savefig(...)` | 接收返回的 Axes 后使用相同方法 |

Pandas 还提供 `.plot.pie()`、`.plot.kde()` 和 `pandas.plotting.scatter_matrix()` 等入口。
饼图适合少量互斥类别的组成；类别多时可以用柱形图。KDE 是核密度估计，需要 SciPy，
平滑程度受带宽影响，尤其不能把整数发帖数的平滑曲线当成实际观测频数。
散点矩阵适合少量数值变量的探索；相关性仍不能直接解释成因果关系。

### 常见显示问题

| 现象 | 检查与处理 |
|---|---|
| 中文显示成方框 | 核对已安装字体与开场打印结果，重跑字体设置和图形单元 |
| 长标签互相遮挡 | 用横向柱形图、增大 figsize 或旋转刻度标签 |
| 日期顺序不对 | 先将日期转换成时间类型并排序，确认本地时区 |
| 图中出现奇怪折线 | 看横轴是否有顺序意义、是否将不同用户误连为时间序列 |
| 导出空白图 | 保留 fig 对象，在 show 前使用 fig.savefig |
| 重复运行出现旧图或太多画布 | 每格创建新 fig；已保存且不再使用的图可 plt.close(fig) |

### 参考材料与版本

- 本地 `references/plots/quick_start.ipynb`：参考 Figure / Axes、接口、样式、刻度、色条与布局。
- 本地 `references/plots/pyplot.ipynb`：参考 pyplot、类别图、子图、文字、注释与坐标尺度。
- [Matplotlib Quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html)
  与 [Pyplot tutorial](https://matplotlib.org/stable/tutorials/pyplot.html)。
- 用户指定的 [Pandas 绘图入门](https://pandas.pydata.org/docs/getting_started/intro_tutorials/04_plotting.html)
  与 [Pandas visualization 用户指南](https://pandas.pydata.org/docs/user_guide/visualization.html)。

示例与中文解释围绕本课数据重新编写。参考网页可能对应更新版本，本教程采用课程环境中实际运行的接口，
实际 Pandas、Matplotlib 和 Plotly 版本在相应单元打印。Plotly 演示纳入执行验证，
hvPlot 与 Altair 仍为官方文档写法介绍，不计入代码执行验证。